# Defter 3 (SON) — Değerlendirme: Analiz Planı'ndaki tüm ölçütler
**Embedding tree growth theory into AI** · Prof. Dr. İlker Ercanlı

**Önce:** Defter 2 (SON) bitmiş olmalı (`ITRDB_AI/SONUC_SON/`; model içi analizler dahil). GPU gerekmez; "Yüksek RAM" önerilir.
Çıktılar `ITRDB_AI/ANALIZ_SON/` klasörüne yazılır. Süre: 20–40 dk.

Tablolar: T1 ana tablo · T1b duyarlılık (yükselti bayraklı alanlar hariç) · T2 yenilik dilimleri · T3 hata–yenilik eğimi (H2, H3)
· T4 ablasyon · T5 metabolik üs · T7 serbest gidiş · T8 m profili · T9 teori ihlali (H5) · T10–T11 SHAP · T12 kurak teşhis
· T13 yetişme ortamı etkisi · T14 hata varyansı ayrıştırması (H4) · T15 kronoloji korelasyonu · T16 kalibre edilmiş R²

In [ ]:
import os, json, glob, math, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import norm
try:
    from google.colab import drive
    drive.mount('/content/drive'); ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:
    ANA = os.environ.get('ITRDB_AI', '.')
SONUC = os.path.join(ANA, 'SONUC_SON'); MA = os.path.join(SONUC, 'MODEL_ANALIZ')
ANALIZ = os.path.join(ANA, 'ANALIZ_SON'); os.makedirs(ANALIZ, exist_ok=True)
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz')); soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']
k = d['ring_year'] > YIL0                                   # Defter 2 ile aynı süzme
R_ser = d['ring_series'][k]; R_yil = d['ring_year'][k].astype(int); R_w = d['ring_rw'][k]
site = d['series_site'][R_ser]; tree = d['series_tree'][R_ser]
Y = np.where(R_w > 0, np.log(np.maximum(R_w, 1e-6)), np.log(0.005)); NZ = R_w > 0
bayrak = d['site_yukselti_bayrak']
pd.set_option('display.width', 220)
print(f'{len(R_w):,} halka yüklendi')

isler = []
for f in sorted(glob.glob(os.path.join(SONUC, '*', 'bitti.json'))):
    j = json.load(open(f)); b, m, s = j['is_ad'].split('__')
    isler.append(dict(bolunme=b, model=m, tohum=int(s[1:]), klasor=os.path.dirname(f), **{x: j[x] for x in ['rmse_log', 'bias_log', 'nll', 'kapsama90', 'n_test', 'en_iyi_dogrulama']}))
isler = pd.DataFrame(isler)
print(len(isler), 'bitmiş iş'); print(isler.groupby(['bolunme', 'model']).size().unstack().fillna(0).astype(int).to_string())

def topluluk(bolunme, model):
    """Tohumları birleştir (deep ensemble): μ̄ = ort(μ_k), σ² = ort(σ_k²) + var(μ_k)"""
    sec = isler[(isler.bolunme == bolunme) & (isler.model == model)]
    MU, V, idx = [], [], None
    for kl in sec.klasor:
        z = np.load(os.path.join(kl, 'test_tahmin.npz'))
        if idx is None: idx = z['idx']
        assert np.array_equal(idx, z['idx'])
        MU.append(z['mu'].astype(np.float32)); V.append(z['sigma'].astype(np.float32) ** 2)
    MU, V = np.stack(MU), np.stack(V)
    return idx, MU.mean(0), np.sqrt(V.mean(0) + MU.var(0)), len(sec)

def olcutler(y, mu, sg, nz):
    r = (y - mu)[nz]; z = (y - mu) / sg
    nll = np.where(nz, 0.5 * z**2 + np.log(sg) + 0.9189385, -norm.logcdf((math.log(0.01) - mu) / sg))
    crps = sg * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / math.sqrt(math.pi))
    return dict(RMSE=float(np.sqrt((r**2).mean())), bias=float(r.mean()), NLL=float(nll.mean()),
                CRPS=float(crps[nz].mean()), kapsama90=float((np.abs(z) < 1.645)[nz].mean()))

In [ ]:
# --- İklim yeniliği (Mahalanobis; 24 aylık sıcaklık–nem penceresi + alan normalleri) ---
clim = d['clim']; soil = d['soil']; desc = d['desc']
Tm = clim[:, :, :, 0].reshape(len(desc), -1); Mm = soil.reshape(len(desc), -1)
def ozellik(idx):
    s = site[idx]; ay = ((R_yil[idx] - 1 - YIL0) * 12)[:, None] + np.arange(24)[None, :]
    return np.concatenate([Tm[s[:, None], ay], Mm[s[:, None], ay], desc[s][:, 2:]], 1).astype(np.float64)

def egitim_halkalari(bolunme):
    bolge = np.array(soz['site_bolge']); tak = d['site_takson_test']
    if bolunme == 'zaman': return np.where(R_yil <= 1960)[0]
    if bolunme.startswith('bolge-'): return np.where(bolge[site] != bolunme[6:])[0]
    if bolunme == 'takson': return np.where(~tak[site])[0]
    if bolunme.startswith('iklim-'):
        x = desc[:, 2] if bolunme == 'iklim-sicak' else -desc[:, 5]
        return np.where(x[site] < np.quantile(x, 0.9))[0]

rng = np.random.default_rng(0)
def yenilik(bolunme, test_idx):
    tr = egitim_halkalari(bolunme); tr = rng.choice(tr, min(200000, len(tr)), replace=False)
    Xtr = ozellik(tr); mu = Xtr.mean(0); Ci = np.linalg.inv(np.cov(Xtr, rowvar=False) + 1e-6 * np.eye(Xtr.shape[1]))
    ref = np.quantile(np.sqrt(np.einsum('ij,jk,ik->i', Xtr - mu, Ci, Xtr - mu)), 0.99)
    out = np.empty(len(test_idx))
    for i in range(0, len(test_idx), 500000):
        X = ozellik(test_idx[i:i+500000]) - mu; out[i:i+500000] = np.sqrt(np.einsum('ij,jk,ik->i', X, Ci, X)) / ref
    return out

def egim_bootstrap(x, e_t, e_s, st, B=1000):
    """|hata| ~ yenilik eğimleri ve farkın alan düzeyinde bootstrap GA'sı (alan başına yeterli istatistiklerle, hızlı)"""
    us, inv = np.unique(st, return_inverse=True)
    def ist(e): return np.stack([np.bincount(inv, w, len(us)) for w in [np.ones_like(x), x, e, x * e, x * x]], 1)
    St, Ss = ist(e_t), ist(e_s)
    def egim(S): n, sx, sy, sxy, sxx = S.T; return (n * sxy - sx * sy) / (n * sxx - sx ** 2)
    W = rng.multinomial(len(us), np.full(len(us), 1 / len(us)), size=B).astype(float)
    fark = egim((W @ Ss).T.T) - egim((W @ St).T.T)
    return egim(St.sum(0)[None])[0], egim(Ss.sum(0)[None])[0], np.quantile(fark, [0.025, 0.975])

In [ ]:
# --- T1, T1b, T2, T3, T13: ana karşılaştırmalar ---
ANA_B = [b for b in ['zaman', 'takson', 'bolge-North America', 'bolge-Europe', 'bolge-Asia', 'bolge-Southern Hemisphere', 'iklim-sicak', 'iklim-kurak']
         if ((isler.bolunme == b) & isler.model.isin(['serbest', 'serbest_ortam'])).any()]
MODELLER = ['serbest', 'teori', 'teori_vpd', 'serbest_ortam', 'teori_ortam', 'teori_vpd_ortam']
ref_model = lambda m: 'serbest_ortam' if m.endswith('_ortam') else 'serbest'
tablo, tablo_b, bin_satir, egim, TOPL = [], [], [], [], {}
for b in ANA_B:
    var = [m for m in MODELLER if ((isler.bolunme == b) & (isler.model == m)).any()]
    if not var: continue
    T = {m: topluluk(b, m) for m in var}; TOPL[b] = T
    idx = next(iter(T.values()))[0]; y, nz = Y[idx], NZ[idx]; temiz = ~bayrak[site[idx]]
    for m in var:
        _, mu, sg, n = T[m]
        tablo.append(dict(bolunme=b, model=m, tohum=n, n_halka=len(idx), **olcutler(y, mu, sg, nz)))
        tablo_b.append(dict(bolunme=b, model=m, n_halka=int(temiz.sum()), **olcutler(y[temiz], mu[temiz], sg[temiz], nz[temiz])))
    nov = yenilik(b, idx)
    q = np.quantile(nov, np.linspace(0, 1, 11)); bn = np.clip(np.searchsorted(q, nov, side='right') - 1, 0, 9)
    for jj in range(10):
        w = bn == jj
        for m in var:
            _, mu, sg, _ = T[m]
            bin_satir.append(dict(bolunme=b, model=m, dilim=jj, yenilik_medyan=float(np.median(nov[w])), n=int(w.sum()), **olcutler(y[w], mu[w], sg[w], nz[w])))
    for m in [v for v in var if not v.startswith('serbest')]:
        r = ref_model(m)
        if r not in T: continue
        e_t, e_s = np.abs(y - T[m][1])[nz], np.abs(y - T[r][1])[nz]
        gt, gs, ga = egim_bootstrap(nov[nz], e_t, e_s, site[idx][nz])
        egim.append(dict(bolunme=b, teori_modeli=m, karsilastirma=r, egim_teori=gt, egim_serbest=gs, fark_serbest_eksi_teori=gs - gt,
                         fark_GA95_alt=ga[0], fark_GA95_ust=ga[1], yeni_oran=float((nov > 1).mean())))
    print('tamam:', b)
tablo, tablo_b, bins, egim = map(pd.DataFrame, [tablo, tablo_b, bin_satir, egim])
tablo.to_csv(os.path.join(ANALIZ, 'T1_ana_tablo.csv'), index=False); tablo_b.to_csv(os.path.join(ANALIZ, 'T1b_duyarlilik.csv'), index=False)
bins.to_csv(os.path.join(ANALIZ, 'T2_yenilik_dilimleri.csv'), index=False); egim.to_csv(os.path.join(ANALIZ, 'T3_hata_yenilik_egimi.csv'), index=False)
for c in ['RMSE', 'NLL', 'kapsama90']:
    print(f'\n{c}'); print(tablo.pivot(index='bolunme', columns='model', values=c).round(3).to_string())
print('\nH2/H3 — hata–yenilik eğimi farkı (serbest − teori; >0 teori lehine):'); print(egim.round(3).to_string(index=False))
# T13: yetişme ortamı bilgisinin etkisi (aynı model, ortamlı − ortamsız)
t13 = []
for b in ANA_B:
    for m in ['serbest', 'teori', 'teori_vpd']:
        a, o = tablo[(tablo.bolunme == b) & (tablo.model == m)], tablo[(tablo.bolunme == b) & (tablo.model == m + '_ortam')]
        if len(a) and len(o): t13.append(dict(bolunme=b, model=m, dRMSE=o.RMSE.iloc[0] - a.RMSE.iloc[0], dNLL=o.NLL.iloc[0] - a.NLL.iloc[0], dKapsama=o.kapsama90.iloc[0] - a.kapsama90.iloc[0]))
t13 = pd.DataFrame(t13); t13.to_csv(os.path.join(ANALIZ, 'T13_ortam_etkisi.csv'), index=False)
print('\nT13 — yetişme ortamı eklenince değişim (negatif RMSE/NLL = iyileşme):'); print(t13.round(4).to_string(index=False))

In [ ]:
# --- Şekil 2: hata ve kapsama, iklim yeniliğine göre ---
fig, ax = plt.subplots(2, len(ANA_B), figsize=(3.4 * len(ANA_B), 7), sharey='row')
renk = {'teori': '#1f6f50', 'teori_vpd': '#2b5fa8', 'serbest': '#c2652a'}
for i, b in enumerate(ANA_B):
    for m in MODELLER:
        z = bins[(bins.bolunme == b) & (bins.model == m)]
        if z.empty: continue
        stil = '--' if m.endswith('_ortam') else '-'; rk = renk[m.replace('_ortam', '')]
        ax[0, i].plot(z.yenilik_medyan, z.RMSE, 'o' + stil, color=rk, label=m, ms=3)
        ax[1, i].plot(z.yenilik_medyan, z.kapsama90, 'o' + stil, color=rk, label=m, ms=3)
    ax[0, i].set_title(b.replace('bolge-', '')); ax[1, i].axhline(0.9, color='grey', lw=0.8, ls=':')
    ax[1, i].set_xlabel('iklim yeniliği')
ax[0, 0].set_ylabel('RMSE (log halka genişliği)'); ax[1, 0].set_ylabel('%90 aralık kapsaması'); ax[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(ANALIZ, 'Sekil2_yenilik.png'), dpi=150); plt.show()

In [ ]:
# --- T14: hata varyansının ayrıştırılması (H4) · T15: kronoloji korelasyonu · T16: kalibre edilmiş R² (zamansal test) ---
t14, t15, t16 = [], [], []
if 'zaman' in TOPL:
    for m, (idx, mu, sg, n) in TOPL['zaman'].items():
        nz = NZ[idx]; i2 = idx[nz]; r = (Y[i2] - mu[nz]); s_, t_, yl = site[i2], tree[i2], R_yil[i2]
        df = pd.DataFrame(dict(r=r, s=s_, t=t_))
        ms = df.groupby('s').r.transform('mean'); mt = df.groupby('t').r.transform('mean')
        t14.append(dict(model=m, toplam_var=float(r.var()), alanlar_arasi=float(ms.var()), alan_ici_agaclar_arasi=float((mt - ms).var()),
                        agac_ici_yillar=float((r - mt).var()), ort_hata=float(r.mean())))
        k = pd.DataFrame(dict(s=s_, yil=yl, y=Y[i2], p=mu[nz])).groupby(['s', 'yil']).mean().reset_index()
        rr = k.groupby('s').filter(lambda g: len(g) >= 20).groupby('s').apply(lambda g: np.corrcoef(g.y, g.p)[0, 1])
        t15.append(dict(model=m, n_alan=len(rr), kronoloji_r_medyan=float(rr.median()), r_q25=float(rr.quantile(0.25)), r_q75=float(rr.quantile(0.75))))
        ser = R_ser[i2]; kal = yl <= 1970
        ofs = pd.Series(r[kal]).groupby(ser[kal]).agg(['mean', 'size']); ofs = ofs[ofs['size'] >= 5]['mean']
        deg = (yl > 1970) & np.isin(ser, ofs.index)
        yy, pp = Y[i2][deg], mu[nz][deg]; pk = pp + ofs.reindex(ser[deg]).to_numpy()
        sst = ((yy - yy.mean()) ** 2).sum()
        t16.append(dict(model=m, n_halka=int(deg.sum()), R2_kalibresiz=float(1 - ((yy - pp) ** 2).sum() / sst), R2_kalibreli_10yil=float(1 - ((yy - pk) ** 2).sum() / sst)))
for ad, tb in [('T14_hata_varyans_ayristirma', t14), ('T15_kronoloji_korelasyonu', t15), ('T16_kalibre_R2', t16)]:
    tb = pd.DataFrame(tb); tb.to_csv(os.path.join(ANALIZ, ad + '.csv'), index=False); print(f'\n{ad}'); print(tb.round(4).to_string(index=False))

In [ ]:
# --- T4 ablasyon · T5 metabolik üs · T7 serbest gidiş · T8 m profili ---
abl = []
for b in ['zaman', 'takson', 'bolge-Southern Hemisphere', 'iklim-kurak']:
    for m in ['teori_ortam', 'abl_G_ortam', 'abl_min_ortam', 'abl_mono_ortam', 'abl_r0_ortam', 'serbest_ortam']:
        z = isler[(isler.bolunme == b) & (isler.model == m) & (isler.tohum == 0)]
        if len(z): abl.append(dict(bolunme=b, model=m, **z.iloc[0][['rmse_log', 'nll', 'kapsama90']].to_dict()))
abl = pd.DataFrame(abl); abl.to_csv(os.path.join(ANALIZ, 'T4_ablasyon.csv'), index=False); print(abl.round(4).to_string(index=False))

kl_m = isler[(isler.bolunme == 'zaman') & (isler.model == 'teori_ortam')].klasor
if len(kl_m):
    P = [np.load(os.path.join(kl, 'alan_param.npz')) for kl in kl_m]; m_ust = np.mean([p['m'] for p in P], 0)
    genus = np.array(soz['genus'])[d['site_genus']]
    ozet_m = pd.DataFrame({'genus': genus, 'm': m_ust}).groupby('genus').m.agg(['median', 'count']).sort_values('count', ascending=False)
    ozet_m.to_csv(os.path.join(ANALIZ, 'T5_metabolik_us_m.csv'))
    print(f'\nÖğrenilen m (teori_ortam): medyan {np.median(m_ust):.3f}, %90 aralık {np.quantile(m_ust, 0.05):.3f}–{np.quantile(m_ust, 0.95):.3f}')
    om = np.mean([p['omega'] for p in P], 0); print('Ay ağırlıkları (önceki yıl Oca–Ara | bu yıl Oca–Ara):', np.round(om.mean(0), 3))

sg_ = [json.load(open(f)) for f in glob.glob(os.path.join(SONUC, 'zaman__*', 'serbest_gidis.json'))]
if sg_:
    sg_ = pd.DataFrame(sg_); sg_.to_csv(os.path.join(ANALIZ, 'T7_serbest_gidis.csv'), index=False)
    print(sg_.groupby('model')[['rmse_log_halka', 'birikimli_log_oran_rmse', 'birikimli_log_oran_bias']].mean().round(4).to_string())

pr = []
for f in glob.glob(os.path.join(SONUC, 'zaman__teori_mf*_ortam__s0', 'bitti.json')):
    j = json.load(open(f)); kod = j['is_ad'].split('__')[1][8:-6]
    pr.append(dict(m=int(kod) / (1000 if len(kod) == 4 else 100), dogrulama_NLL=j['en_iyi_dogrulama'], test_NLL=j['nll'], test_RMSE=j['rmse_log']))
if pr:
    pr = pd.DataFrame(pr).sort_values('m'); pr.to_csv(os.path.join(ANALIZ, 'T8_m_profil.csv'), index=False); print(pr.round(4).to_string(index=False))
    plt.figure(figsize=(5.5, 3.8)); plt.plot(pr.m, pr.dogrulama_NLL, 'o-', color='#1f6f50', label='doğrulama NLL')
    plt.plot(pr.m, pr.test_NLL, 's--', color='#2b5fa8', label='test NLL'); plt.axvline(2/3, color='#c2652a', ls=':', label='2/3')
    plt.xlabel('sabitlenen üs m'); plt.ylabel('NLL'); plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(ANALIZ, 'Sekil5_m_profil.png'), dpi=150); plt.show()

In [ ]:
# --- T9–T12: model içi analizler (Defter 2'nin son hücrelerinden) ve Şekil 6 (SHAP) ---
for ad in ['T9_teori_ihlal', 'T10_shap_grup', 'T11_shap_parametre', 'T12_kurak_teshis']:
    f = os.path.join(MA, ad + '.csv')
    if os.path.exists(f):
        tb = pd.read_csv(f); tb.to_csv(os.path.join(ANALIZ, ad + '.csv'), index=False); print(f'\n{ad}'); print(tb.round(4).to_string(index=False))
f = os.path.join(MA, 'T10_shap_grup.csv')
if os.path.exists(f):
    sh = pd.read_csv(f).pivot(index='grup', columns='model', values='ort_mutlak_shap')
    sh = sh.loc[sh.mean(1).sort_values().index]
    sh.plot.barh(figsize=(7, 4.5), color=['#c2652a', '#1f6f50', '#2b5fa8'][:sh.shape[1]])
    plt.xlabel('ortalama |SHAP| (log halka genişliği)'); plt.ylabel(''); plt.tight_layout()
    plt.savefig(os.path.join(ANALIZ, 'Sekil6_shap.png'), dpi=150); plt.show()

print('\n==== DEĞERLENDİRME (SON) TAMAM ====  Dosyalar:', ANALIZ)